<a href="https://colab.research.google.com/github/Hanzet22/TKJ-Dumps/blob/main/Projek_Aurellia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y iputils-ping > /dev/null 2>&1
print("✅ Ping installed!")

✅ Ping installed!


In [ ]:
# ================================================
#  📡 PingProbe v1.0
#  Produk TKJ - Karya: Aurellia
#  Fungsi: Ping tester dinamis (manual & auto)
# ================================================

import subprocess
import platform
import time
from datetime import datetime
import ipaddress

print("""
   ╔═══════════════════════════════════════╗
   ║   📡  PingProbe v1.0                 ║
   ║   Dynamic Ping Tester Sederhana      ║
   ╚═══════════════════════════════════════╝
""")

def ping_host(host, count=4, interval=1, timeout=2):
    """Ping host dengan parameter dinamis"""
    param = "-n" if platform.system().lower() == "windows" else "-c"
    timeout_param = "-w" if platform.system().lower() == "windows" else "-W"
    interval_param = "" if platform.system().lower() == "windows" else "-i"

    # Build command
    cmd = ["ping", param, str(count), host]

    # Tambah interval (hanya untuk Linux/Mac)
    if platform.system().lower() != "windows" and interval > 0:
        cmd.extend([interval_param, str(interval)])

    # Timeout (Windows beda)
    if platform.system().lower() == "windows":
        cmd.extend([timeout_param, str(timeout * 1000)])
    else:
        cmd.extend([timeout_param, str(timeout)])

    try:
        output = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout + 2)
        return output.stdout, output.stderr
    except subprocess.TimeoutExpired:
        return "❌ Waktu habis (timeout)", ""

def parse_ping_output(output, host):
    """Parse hasil ping biar rapi"""
    lines = output.split('\n')
    sent, received, lost, min_time, avg_time, max_time = 0, 0, 0, 0, 0, 0

    # Parse untuk Linux/Mac
    if "icmp_seq" in output or "packets transmitted" in output:
        for line in lines:
            if "packets transmitted" in line:
                parts = line.split()
                for i, p in enumerate(parts):
                    if p == "packets":
                        sent = int(parts[i-1])
                    elif p == "received":
                        received = int(parts[i-1])
                    elif p == "packet" and "loss" in parts[i+1]:
                        lost_str = parts[i+2].replace('%', '').replace(',', '')
                        lost = int(lost_str) if lost_str.isdigit() else 0
            if "min/avg/max" in line:
                parts = line.split('=')
                if len(parts) > 1:
                    nums = parts[1].split('/')
                    if len(nums) >= 3:
                        min_time = float(nums[0])
                        avg_time = float(nums[1])
                        max_time = float(nums[2])

    # Parse untuk Windows
    elif "Received" in output:
        for line in lines:
            if "Sent = " in line:
                sent = int(line.split("Sent = ")[1].split(",")[0])
                received = int(line.split("Received = ")[1].split(",")[0])
                lost = int(line.split("Lost = ")[1].split(",")[0])
            if "Minimum = " in line:
                nums = line.replace("Minimum = ", "").replace("ms", "").replace("Maximum = ", "").replace("Average = ", "").split(",")
                if len(nums) >= 3:
                    min_time = float(nums[0].strip())
                    max_time = float(nums[1].strip())
                    avg_time = float(nums[2].strip())

    return sent, received, lost, min_time, avg_time, max_time

# ================================================
# MENU UTAMA
# ================================================
while True:
    print("\n" + "="*50)
    print("📡 PINGPROBE — MENU UTAMA")
    print("="*50)
    print("1. Ping Manual (1x)")
    print("2. Ping Berulang (dynamic count)")
    print("3. Ping Monitoring (terus-menerus)")
    print("4. Ping dengan Interval custom")
    print("5. Cek Hostname/IP")
    print("6. Keluar")

    pilih = input("\nPilih menu (1-6): ").strip()

    if pilih in ["1", "2", "3", "4"]:
        host = input("🎯 Masukkan IP/Domain: ").strip()
        if not host:
            print("❌ Host tidak boleh kosong!")
            continue

        # Resolve dulu
        try:
            import socket
            resolved = socket.gethostbyname(host)
            print(f"✅ Resolved: {host} → {resolved}")
            target = host
        except:
            print("❌ Gagal resolve domain!")
            continue

        # Count
        if pilih == "1":
            count = 1
            interval = 0
        elif pilih == "2":
            try:
                count = int(input("Jumlah ping (contoh: 10): ").strip() or "4")
            except:
                count = 4
            interval = 0
        elif pilih == "3":
            count = 0  # infinite
            interval = 1
        elif pilih == "4":
            try:
                count = int(input("Jumlah ping: ").strip() or "4")
                interval = float(input("Interval (detik, contoh: 0.5): ").strip() or "1")
            except:
                count = 4
                interval = 1

        print(f"\n⏳ Ping ke {target}...")
        if count > 0:
            output, err = ping_host(target, count, interval)
            print("\n" + output)
            if err:
                print("⚠️ Error:", err)

            # Parse & tampilkan ringkasan
            sent, received, lost, min_t, avg_t, max_t = parse_ping_output(output, target)
            if sent > 0:
                print("\n📊 RINGKASAN:")
                print(f"   Paket dikirim: {sent}")
                print(f"   Paket diterima: {received}")
                print(f"   Paket hilang: {lost} ({lost/sent*100:.1f}%)")
                if avg_t > 0:
                    print(f"   Min: {min_t:.0f} ms, Avg: {avg_t:.0f} ms, Max: {max_t:.0f} ms")
            else:
                print("❌ Tidak ada respons dari host.")
        else:
            # Mode monitoring (infinite)
            print(f"\n🔄 Monitoring {target} (tekan Ctrl+C untuk berhenti)...\n")
            try:
                no = 1
                while True:
                    output, _ = ping_host(target, 1, interval)
                    now = datetime.now().strftime("%H:%M:%S")

                    # Cek status
                    if "time=" in output or "TTL" in output or "Reply from" in output:
                        # Ambil waktu ping
                        if "time=" in output:
                            time_part = output.split("time=")[1].split(" ")[0]
                        elif "time<" in output:
                            time_part = output.split("time<")[1].split(" ")[0]
                        else:
                            time_part = "N/A"
                        status = f"✅ UP ({time_part} ms)"
                    else:
                        status = "❌ DOWN"

                    print(f"[{now}] #{no} → {status}")
                    no += 1
                    time.sleep(interval)
            except KeyboardInterrupt:
                print("\n⏹️ Monitoring dihentikan.")

    elif pilih == "5":
        host = input("🎯 Masukkan IP/Domain: ").strip()
        try:
            import socket
            resolved = socket.gethostbyname(host)
            try:
                hostname = socket.gethostbyaddr(resolved)[0]
                print(f"📡 {host} → {resolved} → {hostname}")
            except:
                print(f"📡 {host} → {resolved}")
        except:
            print("❌ Gagal resolve.")

    elif pilih == "6":
        print("\n👋 Sampai jumpa! PingProbe siap sedia.")
        break

    else:
        print("❌ Pilihan tidak valid.")